# Week 3: Search Intelligence Data Contract

**Lane:** CTR / Engagement Opportunity Scoring

This contract defines a page-level search opportunity slice: visible pages that may be under-capturing clicks and deserve metadata or content review. The working month is March 2026, a mid-panel month rather than the sealed June 2026 sample month.

## 1) Contract in plain words

1. **What one row means:** One row in the raw fact is one pseudonymized content item for one pseudonymized client on one report date; the decision frame aggregates this to one client-content-month.
2. **Which table(s):** I use `fact_content_daily_performance` for Search Console and Analytics signals, with `dim_content` available for content metadata when needed.
3. **Which time window:** The feature window is the real `month=2026-03` partition; April is held out as the future outcome window for the leakage demonstration.
4. **What I predict or rank:** I score client-content-month rows by CTR opportunity, using future CTR or click improvement as the eventual label and current CTR versus position as the initial proxy.
5. **What I deliberately exclude:** I exclude April outcomes and all label-derived fields from the decision-time feature frame.

## 2) Load the real lane slice

This notebook reads the downloaded, gated FlyRank warehouse release directly from parquet. The raw fact is daily; the lane decision frame aggregates available Search Console rows to client-content-month. Hash keys are retained only for grouping and review joins, never for public reporting.

In [7]:
from pathlib import Path
import duckdb

workspace_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / 'data' / 'internship-warehouse').exists()),
    Path.cwd(),
)
DATA_ROOT = workspace_root / 'data' / 'internship-warehouse'
MONTH = '2026-03'
connection = duckdb.connect()
daily_path = DATA_ROOT / 'fact_content_daily_performance' / f'month={MONTH}' / '*.parquet'
future_path = DATA_ROOT / 'fact_content_daily_performance' / 'month=2026-04' / '*.parquet'
connection.execute(f"CREATE OR REPLACE VIEW daily AS SELECT * FROM read_parquet('{daily_path.as_posix()}')")

page_month = connection.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)::DOUBLE AS impressions,
        SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        SUM(gsc_sum_position)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
        SUM(sessions_organic)::DOUBLE AS sessions_organic,
        SUM(ga4_engaged_sessions)::DOUBLE / NULLIF(SUM(ga4_sessions), 0) AS engaged_session_rate
    FROM daily
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f'Real partition loaded: {daily_path}')
print(f'Page-month rows: {len(page_month):,}')
page_month.head()

Real partition loaded: c:\Users\Brianna\Desktop\FLYRANK\data\internship-warehouse\fact_content_daily_performance\month=2026-03\*.parquet
Page-month rows: 176,738


,client_hash_id,content_hash_id,impressions,ctr,avg_position,sessions_organic,engaged_session_rate
0,client_73cda7b4e4f265ea,content_051a6b43ebe7f09d,186.0,0.0,66.005376,0.0,NaN
1,client_73cda7b4e4f265ea,content_e0e2088ee79c773b,215.0,0.0,0.260465,0.0,NaN
2,client_73cda7b4e4f265ea,content_ca2c5289ed4504fd,209.0,0.0,47.645933,0.0,NaN
3,client_73cda7b4e4f265ea,content_0aa38c3d8bb651b5,164.0,0.0,30.079268,0.0,NaN
4,client_73cda7b4e4f265ea,content_f8deab805147cd7c,1458.0,0.0,25.319616,0.0,NaN


## 3) Three verification queries

These are the only three verification queries in this notebook. Query 1 checks the grain, Query 2 checks the slice size and date span, and Query 3 checks availability with `IS TRUE`.

In [8]:
# Query 1: prove the raw fact grain is one client-content-report-date.
q1 = """
SELECT COUNT(*) AS grain_violations
FROM (
  SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
  FROM daily
  GROUP BY report_date, client_hash_id, content_hash_id
  HAVING COUNT(*) <> 1
)
"""
q1_result = connection.sql(q1).df()
print(f"Q1 raw grain violations: {q1_result.loc[0, 'grain_violations']}")

# Query 2: prove the real March slice row count and date span.
q2 = """
SELECT COUNT(*) AS rows_in_slice, MIN(report_date) AS first_date, MAX(report_date) AS last_date
FROM daily
"""
q2_result = connection.sql(q2).df()
print(
    f"Q2 raw rows: {q2_result.loc[0, 'rows_in_slice']:,}; "
    f"date span: {q2_result.loc[0, 'first_date']} to {q2_result.loc[0, 'last_date']}"
)

# Query 3: prove availability using the required IS TRUE filter.
q3 = """
SELECT COUNT(*) AS available_rows
FROM daily
WHERE gsc_data_available IS TRUE
"""
q3_result = connection.sql(q3).df()
print(f"Q3 available rows after IS TRUE: {q3_result.loc[0, 'available_rows']:,}")

Q1 raw grain violations: 0
Q2 raw rows: 9,841,378; date span: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
Q3 available rows after IS TRUE: 3,611,061


## 4) Five features and the leakage trap

The feature frame contains five features maximum. Every feature is knowable when the review queue is generated. `expected_ctr` is a position-based benchmark, and `ctr_gap` is the opportunity signal.

In [9]:
# Five decision-time features maximum.
features = page_month[['client_hash_id', 'content_hash_id']].copy()
features['impressions'] = page_month['impressions']
features['ctr'] = page_month['ctr']
features['avg_position'] = page_month['avg_position']
features['sessions_organic'] = page_month['sessions_organic']
expected_ctr = 0.10 / features['avg_position'].clip(lower=1).pow(0.35)
features['ctr_gap'] = expected_ctr - features['ctr']

print(f'Feature frame shape: {features.shape}')
print('Available when?')
print('- impressions: known from Search Console when the March feature partition closes.')
print('- ctr: calculated from March clicks and impressions before the review decision.')
print('- avg_position: known from March Search Console observations.')
print('- sessions_organic: known from March Analytics observations when GA4 is available.')
print('- ctr_gap: calculated from current CTR and a fixed position benchmark; no future outcome is used.')
features.head()

# Deliberate leakage experiment: April future_ctr is label-derived and unavailable in March.
connection.execute(f"CREATE OR REPLACE VIEW future_daily AS SELECT * FROM read_parquet('{future_path.as_posix()}')")
future = connection.sql("""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS future_ctr
    FROM future_daily
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()
scored = page_month.merge(future, on=['client_hash_id', 'content_hash_id'], how='inner').dropna(subset=['future_ctr', 'ctr'])
scored['ctr_gap'] = 0.10 / scored['avg_position'].clip(lower=1).pow(0.35) - scored['ctr']
scored['future_ctr_improved'] = (scored['future_ctr'] > scored['ctr']).astype(int)
honest_prediction = (scored['ctr_gap'] > features['ctr_gap'].median()).astype(int)
leaky_prediction = (scored['future_ctr'] > scored['ctr']).astype(int)
honest_score = (honest_prediction == scored['future_ctr_improved']).mean()
leaky_score = (leaky_prediction == scored['future_ctr_improved']).mean()
print(f'Honest quick score: {honest_score:.3f}')
print(f'Leaky quick score: {leaky_score:.3f}')

# Remove the future label and all label-derived experiment columns.
scored = scored.drop(columns=['future_ctr', 'future_ctr_improved'])
print(f"future_ctr removed: {'future_ctr' not in scored.columns}")

Feature frame shape: (176738, 7)
Available when?
- impressions: known from Search Console when the March feature partition closes.
- ctr: calculated from March clicks and impressions before the review decision.
- avg_position: known from March Search Console observations.
- sessions_organic: known from March Analytics observations when GA4 is available.
- ctr_gap: calculated from current CTR and a fixed position benchmark; no future outcome is used.
Honest quick score: 0.535
Leaky quick score: 1.000
future_ctr removed: True


## 5) Limitation

The main limitation is that a CTR gap does not prove that a page's title, description, or content is the cause of under-capture. Position, query intent, SERP features, seasonality, and missing GA4 coverage can all affect engagement. Hash keys are pseudonymous and must not be used to identify clients or content in public outputs. The score should therefore prioritize human review, not automatically rewrite metadata.

## 6) Self-check

- Five contract answers are stated in plain words.
- Exactly three verification queries are shown against the downloaded warehouse.
- Availability uses `gsc_data_available IS TRUE`.
- The feature frame has five decision-time features maximum.
- Every feature has an `available when?` explanation.
- The April future-label trap is shown, scores higher, and is removed.
- One named limitation is recorded.

**Honest conclusion:** March CTR opportunity is a useful prioritization proxy, but future-month click or CTR improvement is required to evaluate whether the ranking actually helps.